In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import json
import time
import networkx as nx
import faiss
import pickle
import warnings
from collections import defaultdict
from itertools import combinations
warnings.filterwarnings("ignore")

df = pd.read_parquet("/kaggle/input/notebooks/anymansince2005/baseline-model/arxiv_ml_with_corpus.parquet")
df = df.reset_index(drop=True)
embeddings = np.load("/kaggle/input/notebooks/anymansince2005/semantic-embedding/embeddings.npy")
faiss_index = faiss.read_index("/kaggle/input/notebooks/anymansince2005/semantic-embedding/faiss_index.bin")
with open("/kaggle/input/notebooks/anymansince2005/semantic-embedding/all_metrics.json") as f:
    all_metrics = json.load(f)

print(f"Loaded {len(df):,} papers")
print(f"Embeddings shape : {embeddings.shape}")
print(f"FAISS total      : {faiss_index.ntotal:,}")

Loaded 110 papers
Embeddings shape : (110, 384)
FAISS total      : 110


In [3]:
print("\nCreating citation graph ...")
start = time.time()

G = nx.DiGraph()

for idx, row in df.iterrows():
    G.add_node(
        idx,
        paper_id = row["paper_id"],
        title = row["title"],
        categories = row["categories"],
        date = str(row["date"]),
    )
    
print(f"Nodes added: {G.number_of_nodes():,}")

def build_semantic_citaion_edges(
    embeddings : np.ndarray,
    faiss_index,
    df : pd.DataFrame,
    G : nx.DiGraph,
    neighbors_k : int=15,
    sim_threshold : float = 0.82,
):
    edge_count = 0
    batch_size = 110
    
    for start_idx in range(0, len(df), batch_size):
        end_idx = min(start_idx+batch_size, len(df))
        batch_vecs = embeddings[start_idx:end_idx].astype(np.float32)
        scores, indices = faiss_index.search(batch_vecs, neighbors_k+1)
        
        for local_i, (nbr_indices, nbr_scores) in enumerate(zip(indices, scores)):
            global_i = start_idx + local_i
            date_i = df.iloc[global_i]["date"]
            
            for nbr_idx, score in zip(nbr_indices, nbr_scores):
                if nbr_idx == global_i: continue
                if score<sim_threshold: continue

                date_j = df.iloc[nbr_idx]["date"]
                
                if date_i > date_j:
                    G.add_edge(global_i, nbr_idx, weight=float(score), type="semantic")
                else:
                    G.add_edge(nbr_idx, global_i, weight=float(score), type="semantic")
                edge_count+=1
    return edge_count

def build_cocitaion_edges(
    df : pd.DataFrame,
    G  : nx.DiGraph,
    min_shared: int=2,
):
    cat_to_papers = defaultdict(list)
    for idx,row in df.iterrows():
        for cat in row["categories"]:
            cat_to_papers[cat].append(idx)
    edge_count = 0
    seen = set()
    
    for cat,paper_list in cat_to_papers.items():
        if len(paper_list)>500:
            paper_list = paper_list[:500]

        for i,j in combinations(paper_list, 2):
            pair = (min(i,j), max(i,j))
            if pair in seen: continue
            seen.add(pair)

            shared = len(set(df.iloc[i]["categories"]) & set(df.iloc[j]["categories"]))
            if shared>= min_shared :
                G.add_edge(i,j, weight = shared*0.5, type="cocitation")
                edge_count+=1
    return edge_count

print("\nAdding semantic citation edges ...")
sem_edges = build_semantic_citaion_edges(embeddings, faiss_index, df, G, neighbors_k=15, sim_threshold=0.82)
print(f"Semantic edges added  : {sem_edges:,}")

print("\nAdding co-citation edges ...")
coc_edges = build_cocitaion_edges(df, G, min_shared=2)
print(f"Cocitation edges added: {coc_edges:,}")

print(f"\nFinal graph           : {G.number_of_nodes():,} nodes | {G.number_of_edges():,} edges")
print(f"Build time            : {time.time()-start:.1f}s")


Creating citation graph ...
Nodes added: 110

Adding semantic citation edges ...
Semantic edges added  : 4

Adding co-citation edges ...
Cocitation edges added: 136

Final graph           : 110 nodes | 138 edges
Build time            : 0.3s


In [4]:
print("\nComputing graph features ...")

in_degree = dict(G.in_degree())
pagerank = nx.pagerank(G, alpha=0.85, max_iter=100, weight="weight")
hits_hubs, hits_auth = nx.hits(G, max_iter=100, normalized=True)

df["graph_in_degree"] = df.index.map(lambda i: in_degree.get(i,0))
df["graph_pagerank"] = df.index.map(lambda i: pagerank.get(i,0.0))
df["graph_authority"] = df.index.map(lambda i: hits_auth.get(i,0.0))
df["graph_hub"] = df.index.map(lambda i: hits_hubs.get(i,0.0))

print("\nGraph feature stats:")
for col in ["graph_in_degree","graph_pagerank","graph_authority","graph_hub"]:
    print(f"{col:<22} mean = {df[col].mean():.6f} max = {df[col].max():.6f}")

print("\nTop 10 papers by PageRank :")
top_pr = df.nlargest(10,"graph_pagerank")[["title","graph_pagerank","graph_in_degree"]]
print(top_pr.to_string(index=False))


Computing graph features ...

Graph feature stats:
graph_in_degree        mean = 1.254545 max = 10.000000
graph_pagerank         mean = 0.009091 max = 0.048005
graph_authority        mean = 0.009091 max = 0.117441
graph_hub              mean = 0.009091 max = 0.127524

Top 10 papers by PageRank :
                                                                                                                                    title  graph_pagerank  graph_in_degree
                                                                          On complexity of optimized crossover for binary representations        0.048005               10
                                                                     A Novel Model of Working Set Selection for SMO Decomposition Methods        0.029403                5
                                                                                                  Mixed membership stochastic blockmodels        0.029327                5
                  

In [5]:
def cocitation_similarity(G, paper_idx, top_n=50):
    citers = set(G.predecessors(paper_idx))
    if not citers:
        return {}

    co_counts = defaultdict(int)
    for citer in citers:
        for cited in G.successors(citer):
            if cited != paper_idx:
                co_counts[cited] += 1

    total = len(citers)
    return {k: v / total for k, v in
            sorted(co_counts.items(), key=lambda x: -x[1])[:top_n]}

In [6]:
def normalize_series(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mn) / (mx - mn)

df["pr_norm"]   = normalize_series(df["graph_pagerank"])
df["auth_norm"] = normalize_series(df["graph_authority"])
df["ind_norm"]  = normalize_series(df["graph_in_degree"])

df["graph_score"] = (
    0.50 * df["pr_norm"] +
    0.30 * df["auth_norm"] +
    0.20 * df["ind_norm"]
)


def blended_recommend(
    query_idx       : int,
    embeddings      : np.ndarray,
    faiss_index,
    G               : nx.DiGraph,
    df              : pd.DataFrame,
    top_n           : int = 10,
    alpha           : float = 0.70,
    beta            : float = 0.20,
    gamma           : float = 0.10,) -> pd.DataFrame:
    assert abs(alpha + beta + gamma - 1.0) < 1e-6, "Weights must sum to 1"

    query_vec        = embeddings[query_idx].reshape(1, -1).astype(np.float32)
    k_fetch          = min(top_n * 5, len(df))
    sem_scores, idxs = faiss_index.search(query_vec, k_fetch)

    candidate_indices = [int(i) for i in idxs[0] if i != query_idx]
    sem_score_map     = {
        int(i): float(s)
        for i, s in zip(idxs[0], sem_scores[0])
        if i != query_idx
    }

    cocit_map = cocitation_similarity(G, query_idx, top_n=100)

    rows = []
    for cand_idx in candidate_indices:
        sem   = sem_score_map.get(cand_idx, 0.0)
        graph = float(df.iloc[cand_idx]["graph_score"])
        cocit = cocit_map.get(cand_idx, 0.0)

        blended = alpha * sem + beta * graph + gamma * cocit

        rows.append({
            "cand_idx"        : cand_idx,
            "semantic_score"  : round(sem,     4),
            "graph_score"     : round(graph,   4),
            "cocitation_score": round(cocit,   4),
            "blended_score"   : round(blended, 4),
        })

    result_df = pd.DataFrame(rows).sort_values("blended_score", ascending=False)
    result_df = result_df.head(top_n).reset_index(drop=True)

    meta = df.iloc[result_df["cand_idx"].tolist()][
        ["paper_id", "title", "categories", "date"]
    ].reset_index(drop=True)

    return pd.concat([meta, result_df.drop(columns="cand_idx")], axis=1)


def blended_recommend_by_text(
    query_text  : str,
    model,
    embeddings  : np.ndarray,
    faiss_index,
    G           : nx.DiGraph,
    df          : pd.DataFrame,
    top_n       : int = 10,
    alpha       : float = 0.70,
    beta        : float = 0.20,
    gamma       : float = 0.10,
) -> pd.DataFrame:
    from sentence_transformers import SentenceTransformer
    query_vec = model.encode(
        [query_text], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)

    k_fetch          = min(top_n * 5, len(df))
    sem_scores, idxs = faiss_index.search(query_vec, k_fetch)

    rows = []
    for i, s in zip(idxs[0], sem_scores[0]):
        graph = float(df.iloc[int(i)]["graph_score"])
        blended = alpha * float(s) + beta * graph   # no cocit for free text

        rows.append({
            "cand_idx"      : int(i),
            "semantic_score": round(float(s), 4),
            "graph_score"   : round(graph, 4),
            "blended_score" : round(blended, 4),
        })

    result_df = (pd.DataFrame(rows)
                 .sort_values("blended_score", ascending=False)
                 .head(top_n)
                 .reset_index(drop=True))

    meta = df.iloc[result_df["cand_idx"].tolist()][
        ["paper_id", "title", "categories", "date"]
    ].reset_index(drop=True)

    return pd.concat([meta, result_df.drop(columns="cand_idx")], axis=1)

In [7]:
print("\n" + "=" * 60)
print("TEST 1 — Blended recommendation by paper index")
print("=" * 60)

sample_idx = 42
print(f"\nQuery: {df.iloc[sample_idx]['title']}")

recs = blended_recommend(sample_idx, embeddings, faiss_index, G, df, top_n=10)
print("\nTop 10 blended recommendations:")
print(recs[["title", "semantic_score", "graph_score", "blended_score"]].to_string())


print("\n" + "=" * 60)
print("TEST 2 — Weight sensitivity analysis")
print("=" * 60)

configs = [
    (1.00, 0.00, 0.00, "Semantic only"),
    (0.70, 0.20, 0.10, "Blended (default)"),
    (0.50, 0.40, 0.10, "Graph-heavy"),
    (0.60, 0.10, 0.30, "Co-citation-heavy"),
]

print(f"\nQuery: {df.iloc[sample_idx]['title']}\n")
for alpha, beta, gamma, label in configs:
    recs_cfg = blended_recommend(
        sample_idx, embeddings, faiss_index, G, df,
        top_n=5, alpha=alpha, beta=beta, gamma=gamma
    )
    print(f"── {label} (α={alpha} β={beta} γ={gamma})")
    for _, row in recs_cfg.iterrows():
        print(f"   [{row['blended_score']:.4f}] {row['title'][:80]}")
    print()


TEST 1 — Blended recommendation by paper index

Query: The Parameter-Less Self-Organizing Map algorithm

Top 10 blended recommendations:
                                                                                                                       title  semantic_score  graph_score  blended_score
0                        Condition Monitoring of HV Bushings in the Presence of Missing Data\n  Using Evolutionary Computing          0.1995       0.7105         0.3817
1                                                                      A Study in a Hybrid Centralised-Swarm Agent Community          0.2628       0.4843         0.3808
2                                                                            A neural network approach to ordinal regression          0.3298       0.2087         0.3476
3                                              Using artificial intelligence for data reduction in mechanical\n  engineering          0.2377       0.3922         0.3448
4                

In [8]:
def precision_at_k(recommended_cats, query_cats, k=10):
    hits = sum(
        1 for cats in recommended_cats[:k]
        if set(cats) & set(query_cats)
    )
    return hits / k


def evaluate_model(recommend_fn, df, sample_size=110, k=10, **kwargs):
    indices    = np.random.choice(len(df), size=110, replace=False)
    precisions = []

    for idx in indices:
        query_cats = df.iloc[idx]["categories"]
        recs       = recommend_fn(idx, **kwargs)
        p_at_k     = precision_at_k(recs["categories"].tolist(), query_cats, k)
        precisions.append(p_at_k)

    return {
        "mean_precision@k"  : round(np.mean(precisions), 4),
        "median_precision@k": round(np.median(precisions), 4),
        "std"               : round(np.std(precisions), 4),
        "k"                 : k,
        "sample_size"       : 110,
    }


print("\n" + "=" * 60)
print("EVALUATION — All models compared")
print("=" * 60)

np.random.seed(42)

def blended_recommend_eval(idx, **kwargs):
    return blended_recommend(
        query_idx = idx,
        embeddings = embeddings,
        faiss_index = faiss_index,
        G = G,
        df = df,
        top_n = 10,
        alpha = 0.70,
        beta = 0.20,
        gamma = 0.10,
    )

blended_metrics = evaluate_model(
    recommend_fn = blended_recommend_eval,
    df           = df,
    sample_size  = 110,
    k            = 10,
)

all_metrics["blended_graph"] = blended_metrics

print(f"\n{'Metric':<25} {'TF-IDF':>12} {'Sent-BERT':>12} {'Blended':>12}")
print("-" * 65)
for key in ["mean_precision@k", "median_precision@k", "std"]:
    v1 = all_metrics["tfidf_baseline"][key]
    v2 = all_metrics["sentence_bert"][key]
    v3 = blended_metrics[key]
    print(f"  {key:<23} {v1:>12} {v2:>12} {v3:>12}")



EVALUATION — All models compared

Metric                          TF-IDF    Sent-BERT      Blended
-----------------------------------------------------------------
  mean_precision@k                0.48       0.5491       0.6164
  median_precision@k               0.5          0.6          0.7
  std                           0.2467       0.2471       0.2725


In [9]:
print("\n" + "=" * 60)
print("SAVING ARTIFACTS")
print("=" * 60)

with open("/kaggle/working/citation_graph.pkl", "wb") as f:
    pickle.dump(G, f)
print("Saved: citation_graph.pkl")

df.to_parquet("/kaggle/working/arxiv_ml_graph_features.parquet", index=False)
print("Saved: arxiv_ml_graph_features.parquet")

with open("/kaggle/working/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print("Saved: all_metrics.json")


SAVING ARTIFACTS
Saved: citation_graph.pkl
Saved: arxiv_ml_graph_features.parquet
Saved: all_metrics.json
